Generates an MLP regressor & tests it. 

Import & set filepaths

In [1]:
from pathlib import Path

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR.joinpath('data')
ONNX_MODEL_DIR = ROOT_DIR.joinpath('onnx_models')
INDICES_DIR = ROOT_DIR.joinpath("saved_indices")

from csv_to_pd import csvToDf
from data_ingestion import setup_split

import matplotlib.pyplot as plt
from sklearn.neural_network._multilayer_perceptron import MLPRegressor


Data ingestion & setting up split

In [ ]:
df = csvToDf("indices_save", INDICES_DIR)
splt = setup_split(splitIndices=df)

Set up param grid to search MLP parameters

In [ ]:
grid:list[tuple[int,int]] = [(a,b) for a in range(80, 160, 20) for b in range (20, 80, 20)]
param_grid = [
  {'hidden_layer_sizes': grid, "n_iter_no_change": [10]}
]

Perform MLP grid search. This cell will hold up your kernel for a while!

In [ ]:
from sklearn_optimiser import SklearnOptimiser
mlp_clr = MLPRegressor()
optimizer = SklearnOptimiser(splt, mlp_clr, param_grid)
optimizer.optimize("grid search", n_jobs=4)

In [ ]:
print(optimizer.getOptimalParameters())

Convert sklearn model to onnx for porting & save model to file

In [ ]:
from skl2onnx import to_onnx
import onnx

onx = to_onnx(optimizer.getOptimalClassifier(), splt.train.to_numpy(flatten=True))
onnx.save(onx, ONNX_MODEL_DIR.joinpath("mlp_model1.onnx"))